In [ ]:
# Phase 8 Screen A, fold 0: per-diagnosis attention over (slot, group) tokens, random
# training windows and EMA weights on c2's six-slot config, each pair differing in one
# thing, each arm scored as a fifth member next to c2+c3+f1+s2.
import glob, os, shutil, sys, time

GIT_SHA = 'df5241f'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

from knee.dataset import PreppedSlotDataset, _SLOT_PLANES
from knee.dicom import SLOTS, SLOTS_V3
from knee.model import KneeModel
from knee.reports import confidence_from_agreement, report_group_key
from knee.train import (ModelEma, differential_param_groups, make_folds, masked_bce_loss,
                        train_one_epoch)
# The stale-src guard, derived from what this run needs. Without packing the
# arms would differ in BatchNorm statistics as well as the table; without the
# v3 plane lookup the second arm dies on its first study.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel._slot_features', KneeModel._slot_features, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('PreppedSlotDataset.__init__', PreppedSlotDataset.__init__,
     ('weights_df', 'label_columns', 'random_groups')),
    ('train_one_epoch', train_one_epoch, ('accumulation_steps', 'ema')),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, '_backbone_features'), 'rsna-knee-src predates slot packing'
assert hasattr(KneeModel, '_group_tokens'), 'rsna-knee-src predates the group_attention head'
print('src carries packing, the group_attention head, random_groups and ModelEma | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'
CONFIDENCE = confidence_from_agreement(_votes, LABEL_COLUMNS).reset_index()
CONFIDENCE = CONFIDENCE.rename(columns={CONFIDENCE.columns[0]: 'StudyInstanceUID'})
print(f'confidence weights: {len(CONFIDENCE)} studies from {len(_votes)} sources; '
      f'share below 0.5 per label '
      f'{(CONFIDENCE[LABEL_COLUMNS] < 0.5).mean().round(2).to_dict()}')

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# The runs: one session, four arms, c2's recipe throughout (efficientnet_b0, six v2
# slots, Adam 1e-4 / OneCycle 3e-4, batch 8, 8 epochs, augment on, steven_v4, gold held
# out). a0, a1 and a3 read the fixed five groups of the nine stored -- prep v2's pixels
# exactly -- and a2 draws five of nine at random for training. Every arm is read at
# five and at nine groups; arrays go to disk before anything is compared.
EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = 'date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,macro_auc,paired_delta,train_minutes,inference_seconds,promoted,gold_macro'
with open(EXPERIMENTS_CSV, 'w') as f:
    print(_HEADER, file=f)

CONFIGS = [
    # name, head, random training windows, EMA
    ('a0_c2_slotattn', 'slot_attention', False, False),
    ('a1_c2_grpattn', 'group_attention', False, False),
    ('a2_c2_grpattn_rndwin', 'group_attention', True, False),
    ('a3_c2_grpattn_ema', 'group_attention', False, True),
]
HYPOTHESIS = {
    'a0_c2_slotattn':
        'Screen A control: c2 (six v2 slots, slot attention, efficientnet_b0, 8 epochs, steven_v4, '
        'packed) reading the fixed five groups of the 9-group artifact -- prep v2 pixel for pixel. '
        'As a fifth member next to c2+c3+f1+s2 it prices one more model of a known config.',
    'a1_c2_grpattn':
        'a1: per-diagnosis attention over every (slot, group) token instead of the mean over a '
        'slot\'s groups. The 9-group reads gained +0.005 to +0.009 with nothing retrained, so the '
        'windows carry signal the mean discards; this head can weigh the window that shows the finding.',
    'a2_c2_grpattn_rndwin':
        'a2: a1 trained on five random groups of the nine stored each epoch (eval at the fixed '
        'five), so the model sees the in-between windows the confirms never trained on.',
    'a3_c2_grpattn_ema':
        'a3: a1 with EMA weights (decay 0.999, one update per optimizer step). Scored twice: EMA '
        'against its own raw weights is the EMA effect; raw against a1 is the run-to-run noise.',
}
SOURCE, FOLD, N_GROUPS, OUT_SIZE, EPOCHS, BATCH = 'steven_v4', 0, 5, 224, 8, 8
READS = (5, 9)
TIME_BUDGET_S = 8 * 3600
t_start = time.time()

def loader(uids, labels_df, n_groups, shuffle=False, augment=False, random_groups=False):
    ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=labels_df, n_groups=n_groups,
                            out_size=OUT_SIZE, slots=slot_names, augment=augment,
                            random_groups=random_groups)
    # four workers: the 9-group artifacts decode 27 slices per slot whatever is read
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=4)

train_labels, eval_labels = LABEL_SOURCES[SOURCE]
tr_uids, va_uids = train_val_split(folds, val_fold=FOLD, exclude_uids=holdout)
tr_uids = [u for u in tr_uids if u in uid_to_npz]
va_uids = [u for u in va_uids if u in uid_to_npz]

def score_and_save(tag, net):
    """Val and gold at both reads, arrays on disk first."""
    reads = {}
    for n_groups in READS:
        y_true, y_pred = evaluate(net, loader(va_uids, eval_labels, n_groups), device=device)
        g_true, g_pred = evaluate(net, loader(gold_uids, gold_labels, n_groups), device=device)
        np.save(f'/kaggle/working/oof_{tag}_g{n_groups}.npy', y_pred)
        np.save(f'/kaggle/working/gold_pred_{tag}_g{n_groups}.npy', g_pred)
        reads[n_groups] = (y_true, y_pred, g_true, g_pred)
    np.save(f'/kaggle/working/y_{tag}.npy', reads[5][0])
    np.save(f'/kaggle/working/gold_true_{tag}.npy', reads[5][2])
    return reads

def window_profile(net):
    """Mean attention weight per group index over the gold studies, so the log says
    whether the head prefers particular depths or spreads over the windows."""
    net.eval()
    total, n = None, 0
    with torch.no_grad():
        for images, mask, _, _ in loader(gold_uids, gold_labels, N_GROUPS):
            w = net.attention_weights(images.to(device), mask.to(device))   # [B, S, G, L]
            per_group = w.sum(dim=1).mean(dim=-1)                           # [B, G]
            total = per_group.sum(dim=0) if total is None else total + per_group.sum(dim=0)
            n += len(images)
    return (total / n).cpu().numpy()

results = []
for name, head, random_groups, use_ema in CONFIGS:
    if time.time() - t_start > TIME_BUDGET_S:
        print(f'time budget spent, skipping {name}')
        break
    print(f'\n=== {name}: {len(tr_uids)} train / {len(va_uids)} val, head {head}, '
          f'{"random" if random_groups else "fixed"} training windows, EMA {use_ema}, {EPOCHS} epochs')
    train_loader = loader(tr_uids, train_labels, N_GROUPS, shuffle=True, augment=True,
                          random_groups=random_groups)
    model = KneeModel(backbone_name='efficientnet_b0', num_labels=12, pretrained=True,
                      head=head).to(device)
    ema = ModelEma(model) if use_ema else None
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-4, epochs=EPOCHS, steps_per_epoch=len(train_loader))

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, device=device, scaler=scaler,
                               scheduler=scheduler, ema=ema)
        if epoch == 0:
            print(f'  peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')
        print(f'  epoch {epoch + 1}/{EPOCHS} loss {loss:.4f} ({(time.time() - t0) / 60:.1f} min)')
    minutes = (time.time() - t0) / 60

    # the EMA arm is scored twice: raw weights are one more a1 run (the in-session
    # noise floor), and EMA against raw is the EMA effect, paired on the same run
    variants = [(name, model)] if ema is None else [(name + '_raw', model), (name, ema.module)]
    for tag, net in variants:
        reads = score_and_save(tag, net)
        y_true, y_pred, g_true, g_pred = reads[5]
        aucs = per_label_auc(y_true, y_pred)
        macro, gold_macro = macro_auc(y_true, y_pred), macro_auc(g_true, g_pred)
        macro9, gold9 = macro_auc(reads[9][0], reads[9][1]), macro_auc(reads[9][2], reads[9][3])
        print(f'  {tag}: val {macro:.4f} (nine groups {macro9:.4f}), GOLD {gold_macro:.4f} '
              f'(nine groups {gold9:.4f}) in {minutes:.1f} min')
        if head == 'group_attention':
            print(f'  {tag}: mean attention per window (gold, five groups) '
                  f'{np.round(window_profile(net), 3).tolist()}')
        results.append({'config': tag, 'val_g5': macro, 'val_g9': macro9, 'gold_g5': gold_macro,
                        'gold_g9': gold9, 'minutes': minutes, 'final_loss': loss})
        log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=tag,
                       hypothesis=HYPOTHESIS[name], fold_set=f'primary_v3_fold{FOLD}', seed=0,
                       per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                       macro_auc=float(macro), paired_delta=float('nan'), train_minutes=minutes,
                       inference_seconds=float('nan'), promoted=False, gold_macro=float(gold_macro))
        torch.save(net.state_dict(), f'/kaggle/working/{tag}.pt')
    del model, optimizer, scaler, scheduler, ema
    torch.cuda.empty_cache()

In [ ]:
# Pairs at the fixed five groups, the five-to-nine read gain per arm, then each arm as
# a fifth member next to c2+c3+f1+s2: the confirm runs' own fold-0 predictions at five
# groups (the pre-registered read), and the TTA kernels' at nine where they are mounted.
from knee.metrics import paired_macro_auc_delta

def saved(kind, tag):
    return np.load(f'/kaggle/working/{kind}_{tag}.npy')

def mounted_one(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    assert len(hits) == 1, f'{pattern}: expected exactly one mounted file, found {hits}'
    return hits[0]

def same(a, b):
    return np.array_equal(np.nan_to_num(a, nan=-1), np.nan_to_num(b, nan=-1))

def show(label, y, a, b):
    d, lo, hi = paired_macro_auc_delta(y, a, b)
    verdict = 'CI excludes zero' if lo > 0 or hi < 0 else 'CI spans zero'
    print(f'  {label:52s} {d:+.4f} [{lo:+.4f}, {hi:+.4f}]  {verdict}')

print(pd.DataFrame(results)[['config', 'val_g5', 'val_g9', 'gold_g5', 'gold_g9', 'final_loss',
                             'minutes']].to_string(index=False))
done = [r['config'] for r in results]

PAIRS = (('a1 group attention minus a0 slot attention', 'a0_c2_slotattn', 'a1_c2_grpattn'),
         ('a2 random training windows minus a1', 'a1_c2_grpattn', 'a2_c2_grpattn_rndwin'),
         ('a3 raw minus a1 (run-to-run noise, same config)', 'a1_c2_grpattn', 'a3_c2_grpattn_ema_raw'),
         ('a3 EMA minus a3 raw (same run)', 'a3_c2_grpattn_ema_raw', 'a3_c2_grpattn_ema'))

CONFIRMED = ('c2_6slot_attn', 'c3_dinov2_2slot', 'f1_convnext_tiny_6slot_attn', 's2_c2_336_b4x2')

def confirm_fold(name):
    """Fold-0 rows of a confirm run's OOF (its fold-0 model, five groups) and that model's gold."""
    uids = pd.read_csv(mounted_one(f'oof_uids_{name}.csv'))['StudyInstanceUID'].astype(str)
    row = {u: i for i, u in enumerate(uids)}
    rows = [row[u] for u in va_uids]
    return (np.load(mounted_one(f'oof_true_{name}.npy'))[rows],
            np.load(mounted_one(f'oof_pred_{name}.npy'))[rows],
            np.load(mounted_one(f'gold_folds_{name}.npy'))[FOLD])

def tta_fold(name):
    """Fold-0 rows of a TTA kernel's nine-group OOF, or None when that kernel is not mounted."""
    hits = sorted(glob.glob(f'/kaggle/input/**/oof_pred_{name}_g9.npy', recursive=True))
    if len(hits) != 1:
        return None
    uids = pd.read_csv(os.path.join(os.path.dirname(hits[0]), 'oof_uids.csv'))['StudyInstanceUID'].astype(str)
    row = {u: i for i, u in enumerate(uids)}
    return np.load(hits[0])[[row[u] for u in va_uids]]

members = {n: confirm_fold(n) for n in CONFIRMED}
for tier, truth, pred, idx in (('val ', 'y', 'oof', 1), ('gold', 'gold_true', 'gold_pred', 2)):
    if not done:
        break
    y = saved(truth, done[0])
    if tier == 'val ':
        assert all(same(y, members[n][0]) for n in CONFIRMED), 'members were scored on different studies'
    print(f'\n{tier} (n={len(y)}), pairs at the fixed five groups')
    for label, a, b in PAIRS:
        if a in done and b in done:
            show(label, y, saved(pred, a + '_g5'), saved(pred, b + '_g5'))
    print(f'{tier} (n={len(y)}), nine groups minus five, per arm')
    for tag in done:
        show(tag, y, saved(pred, tag + '_g5'), saved(pred, tag + '_g9'))
    four = np.mean([members[n][idx] for n in CONFIRMED], axis=0)
    print(f'{tier} (n={len(y)}), c2+c3+f1+s2 at five groups {macro_auc(y, four):.4f}; each arm as a '
          f'fifth member (bar +0.0015, CI above zero; a0 is the in-session price of one more model)')
    for tag in done:
        show(tag, y, four, (4 * four + saved(pred, tag + '_g5')) / 5)

nine = {n: tta_fold(n) for n in CONFIRMED}
have = [n for n, p in nine.items() if p is not None]
if done and have:
    y = saved('y', done[0])
    base = np.mean([nine[n] for n in have], axis=0)
    print(f'\nval (n={len(y)}), nine-group reads: {"+".join(have)} {macro_auc(y, base):.4f}; each arm '
          f'at nine groups added as one more member')
    for tag in done:
        show(tag, y, base, (len(have) * base + saved('oof', tag + '_g9')) / (len(have) + 1))
print(f'\ntotal {(time.time() - t_start) / 3600:.2f} h')